# Iris erkennen – unser erstes neuronales Netz

<img src="images/03-nn-iris.png" width="50%">

Quelle: ChatGPT generiert
- - -

Wir erstellen ein **kleines neuronales Netz**, das eine Iris-Blume anhand von vier Messwerten erkennt.

- **4 Eingaben:** Länge und Breite von Kelchblatt und Blütenblatt (in cm)
- **3 mögliche Ergebnisse:** Setosa, Versicolor oder Virginica
- **15 trainierbare Parameter:** 3 Neuronen × (4 Gewichte + 1 Zusatzwert)

**Ablauf:** Daten laden → Modell erstellen → 500 Epochen trainieren → Gewichte ansehen → Blumen abfragen.

In [ ]:
%%bash
# Optional: nur ausführen, falls Pakete in der aktuellen Jupyter-Umgebung fehlen.
source ~/.jupyter/bin/activate
pip install -U matplotlib numpy scikit-learn pandas tensorflow packaging

## 1. Iris-Daten laden

Wir verwenden bekannte Blumen als Trainingsdaten. Einen Teil behalten wir für die spätere Gegenprobe zurück.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import keras
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

keras.utils.set_random_seed(42)

iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42, stratify=iris.target
)

# Je ein Beispiel der drei Blumenarten anzeigen.
beispiele = pd.DataFrame(iris.data[[0, 50, 100]], columns=iris.feature_names)
beispiele["Blumenart"] = iris.target_names[iris.target[[0, 50, 100]]]
display(beispiele)
print(f"Trainingsdaten: {len(X_train)} Blumen | Testdaten: {len(X_test)} Blumen")

## 2. Modell mit 15 Parametern erstellen

Das Modell erhält **4 Messwerte** und berechnet daraus **3 mögliche Blumenarten**. Die 15 Parameter werden beim Training gelernt.

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(4,)),
    keras.layers.Dense(3, activation="softmax")
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.02),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
print(f"Trainierbare Parameter: {model.count_params()}")

## 3. Modell trainieren – 500 Epochen

**Eine Epoche** bedeutet: Das Modell verarbeitet einmal alle Trainingsdaten und passt seine Gewichte an. Der Verlauf zeigt, wie es dazulernt.

In [ ]:
# 500 Epochen: Die 15 Parameter werden dabei automatisch angepasst.
verlauf = model.fit(X_train, y_train, epochs=500, verbose=0)

plt.figure(figsize=(8, 4))
plt.plot(range(1, 501), verlauf.history["accuracy"])
plt.xlabel("Epoche")
plt.ylabel("Richtig erkannte Blumen (Anteil)")
plt.title("Lernfortschritt")
plt.ylim(0, 1.05)
plt.grid(alpha=0.2)
plt.show()

## 4. Gelernte Gewichte ansehen

Das Netz hat jetzt **12 Gewichte und 3 Zusatzwerte** gelernt. Jede Spalte gehört zu einer Blumenart.

In [ ]:
gewichte, zusatzwerte = model.layers[0].get_weights()
tabelle = pd.DataFrame(
    np.vstack([gewichte, zusatzwerte]),
    index=[*iris.feature_names, "Zusatzwert"],
    columns=iris.target_names
)
display(tabelle.round(2))

## 5. Wie gut erkennt das Modell die Blumen?

Wir prüfen die Erkennung sowohl mit den Trainingsdaten als auch mit den zurückgehaltenen Testdaten.

In [ ]:
_, training = model.evaluate(X_train, y_train, verbose=0)
_, test = model.evaluate(X_test, y_test, verbose=0)
print(f"Trainingsdaten richtig erkannt: {training:.1%}")
print(f"Testdaten richtig erkannt:     {test:.1%}")

## 6. Drei Blumen abfragen und das Ergebnis prüfen

Für jede Blumenart wählen wir **eine Blume aus den Testdaten**. Wir kennen die richtige Art bereits und können damit prüfen, ob das Modell richtig liegt.

In [ ]:
for klasse in (0, 1, 2):
    index = np.where(y_test == klasse)[0][0]
    eingabe = X_test[index:index + 1]
    erkannt = int(np.argmax(model.predict(eingabe, verbose=0)))

    print(f"Messwerte (cm): {eingabe[0]}")
    print(f"Tatsächlich:    {iris.target_names[klasse]}")
    print(f"Modell erkennt: {iris.target_names[erkannt]}")
    print(f"Gegenprobe:     {'KORREKT' if erkannt == klasse else 'FALSCH'}")
    print()

## 7. Eigene Abfrage

Die vier Messwerte kannst du ändern. Das Modell sagt dann eine Blumenart voraus. Für selbst eingegebene Werte gibt es keine automatische Gegenprobe.

In [ ]:
# Eingabe: Kelchblatt-Länge, Kelchblatt-Breite, Blütenblatt-Länge, Blütenblatt-Breite
meine_blume = np.array([[5.1, 3.5, 1.4, 0.2]])

klasse = int(np.argmax(model.predict(meine_blume, verbose=0)))
print(f"Eingabe:          {meine_blume[0]}")
print(f"Erkannte Blumenart: {iris.target_names[klasse]}")